# LIME Evaluation for the Trained PPO Model

This notebook fixes the issues in `LIME.ipynb` (see `models/lime/README.md`) and adds:

1. The exact same raw-feature -> 15-dimension preprocessing used to train the RL models (RobustScaler statistics and one-hot category order copied from `models/rl/preprocessing_params.json`, which was fitted on `research/data/heart_disease_trimmed.csv` and verified byte-for-byte against `research/data/preprocessed_dataset_2026-07-04_07-18-06.csv` -- see `backend/services/preprocessingService.js` for the production equivalent of this logic).
2. LIME explanations generated against the real, loaded PPO model (`trained_models/ppo.pth`), not a placeholder.
3. Performance metrics (accuracy, precision, recall, F1, specificity, ROC-AUC) computed on the held-out test folds from `research/preprocessing/rl_environment.pkl`, the same evaluation already done in `PPO.ipynb` -- reproduced here so this notebook can verify the metrics standing alone.

### 1. Install and Import Libraries

In [ ]:
!pip install lime --quiet

In [ ]:
import os
import pickle
import json

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.distributions import Categorical

import matplotlib.pyplot as plt

from lime.lime_tabular import LimeTabularExplainer

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings("ignore")

### 2. Device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running Device :", DEVICE)

### 3. Canonical Feature Contract

The ten raw canonical features, in the fixed order mirrored in `backend/utils/featureContract.js`. This is what a patient actually enters -- NOT what the model was trained on (that is the 15-dimension representation built in Section 4).

In [ ]:
CANONICAL_FEATURE_ORDER = [
    "thalach", "restecg", "oldpeak", "slope", "age",
    "sex", "cp", "exang", "trestbps", "fbs"
]

FEATURE_LABELS = {
    "thalach": "Maximum heart rate",
    "restecg": "Resting ECG result",
    "oldpeak": "ECG stress-test change",
    "slope": "ECG ST-segment slope",
    "age": "Age",
    "sex": "Sex",
    "cp": "Chest pain type",
    "exang": "Exercise-related chest discomfort",
    "trestbps": "Resting blood pressure",
    "fbs": "Fasting blood sugar indicator",
}

### 4. Preprocessing Parameters (copied from `models/rl/preprocessing_params.json`)

These are the EXACT RobustScaler statistics (median/IQR) fitted on `research/data/heart_disease_trimmed.csv` by `research/preprocessing/fit_preprocessing_params.py`, and the fixed one-hot category order used at training time. Copied here as literal values (rather than re-fitting a scaler in this notebook) so this notebook uses the IDENTICAL preprocessing as the deployed backend (`backend/services/preprocessingService.js`) and the original RL training data -- verified to reproduce `preprocessed_dataset_2026-07-04_07-18-06.csv` exactly (max abs diff = 0, see `backend/tests/preprocessingService.test.js`).

In [ ]:
# RobustScaler statistics: scaled_value = (raw_value - median) / iqr
ROBUST_SCALER_PARAMS = {
    "age":      {"median": 56.0,  "iqr": 12.0},
    "trestbps": {"median": 130.0, "iqr": 22.0},
    "thalach":  {"median": 140.0, "iqr": 38.5},
    "oldpeak":  {"median": 1.0,   "iqr": 1.9},
}

# Categorical encodings (canonical integer code -> meaning), matching the
# integers a patient's raw input already uses in this application.
SEX_MAP = {"Male": 1, "Female": 0}
FBS_MAP = {"True": 1, "False": 0}
EXANG_MAP = {"True": 1, "False": 0}
SLOPE_MAP = {"downsloping": 0, "flat": 1, "upsloping": 2}
CP_CANONICAL = {"typical angina": 0, "atypical angina": 1, "non-anginal": 2, "asymptomatic": 3}
RESTECG_CANONICAL = {"normal": 0, "st-t abnormality": 1, "lv hypertrophy": 2}

# Canonical integer -> one-hot column name (fixed order -- must never be
# reordered once a model has been trained against it).
CP_INT_TO_ONEHOT = {
    0: "cp_typical angina",
    1: "cp_atypical angina",
    2: "cp_non-anginal",
    3: "cp_asymptomatic",
}
RESTECG_INT_TO_ONEHOT = {
    0: "restecg_normal",
    1: "restecg_st-t abnormality",
    2: "restecg_lv hypertrophy",
}

# The exact 15-dimension feature order the RL models were trained on.
TRAINED_FEATURE_ORDER_15DIM = [
    "sex", "fbs", "exang", "age", "trestbps", "thalach", "oldpeak",
    "cp_asymptomatic", "cp_atypical angina", "cp_non-anginal", "cp_typical angina",
    "restecg_lv hypertrophy", "restecg_normal", "restecg_st-t abnormality",
    "slope",
]

print("Loaded preprocessing parameters for:", list(ROBUST_SCALER_PARAMS.keys()))
print("Trained feature order (15-dim):", TRAINED_FEATURE_ORDER_15DIM)

### 5. Preprocessing Function: Raw 10 Features -> Trained 15-Dimension State

This is a direct port of `backend/services/preprocessingService.js`'s `toTrainedRepresentation()` so this notebook applies IDENTICAL preprocessing to a patient's raw input as the production backend does.

In [ ]:
def to_trained_representation(features: dict) -> np.ndarray:
    """
    Convert a dict of the 10 raw canonical features into the 15-dimension
    vector the RL models were trained on.

    Parameters
    ----------
    features : dict
        Must contain the 10 canonical keys, using the same integer
        encodings as the app (cp: 0-3, restecg: 0-2, sex/fbs/exang: 0|1,
        slope: 0-2), and raw (unscaled) values for age/trestbps/thalach/oldpeak.

    Returns
    -------
    np.ndarray of shape (15,), dtype float32, in TRAINED_FEATURE_ORDER_15DIM order.
    """
    vector = {}

    # Pass-through binary/ordinal features
    vector["sex"] = features["sex"]
    vector["fbs"] = features["fbs"]
    vector["exang"] = features["exang"]
    vector["slope"] = features["slope"]

    # RobustScaler-scaled continuous features
    for key in ("age", "trestbps", "thalach", "oldpeak"):
        params = ROBUST_SCALER_PARAMS[key]
        vector[key] = (features[key] - params["median"]) / params["iqr"]

    # One-hot expansion for cp / restecg
    for column in ["cp_asymptomatic", "cp_atypical angina", "cp_non-anginal", "cp_typical angina",
                   "restecg_lv hypertrophy", "restecg_normal", "restecg_st-t abnormality"]:
        vector[column] = 0

    cp_column = CP_INT_TO_ONEHOT.get(features["cp"])
    if cp_column is None:
        raise ValueError(f"Unrecognized cp value: {features['cp']}")
    vector[cp_column] = 1

    restecg_column = RESTECG_INT_TO_ONEHOT.get(features["restecg"])
    if restecg_column is None:
        raise ValueError(f"Unrecognized restecg value: {features['restecg']}")
    vector[restecg_column] = 1

    ordered = [vector[key] for key in TRAINED_FEATURE_ORDER_15DIM]
    return np.array(ordered, dtype=np.float32)


# Quick sanity check against a known row (row 0 of
# research/data/heart_disease_trimmed.csv / preprocessed_dataset_2026-07-04_07-18-06.csv)
_sample_raw = {
    "thalach": 150, "restecg": 2, "oldpeak": 2.3, "slope": 0, "age": 63,
    "sex": 1, "cp": 0, "exang": 0, "trestbps": 145, "fbs": 1,
}
_expected_15dim = [1, 1, 0, 0.5833333333333334, 0.6818181818181818,
                   0.2597402597402597, 0.6842105263157894, 0, 0, 0, 1, 1, 0, 0, 0]

_actual_15dim = to_trained_representation(_sample_raw)
_max_abs_diff = np.max(np.abs(_actual_15dim - np.array(_expected_15dim, dtype=np.float32)))
print("Sanity check max abs diff vs known preprocessed row:", _max_abs_diff)
assert _max_abs_diff < 1e-6, "Preprocessing does not match the original training data!"
print("Preprocessing verified against preprocessed_dataset_2026-07-04_07-18-06.csv row 0.")

### 6. Load the Trained PPO Model

In [ ]:
# Hyperparameters matching PPO.ipynb (see cell defining HIDDEN_DIM / ACTION_SIZE there)
HIDDEN_DIM = 128
ACTION_SIZE = 2
STATE_SIZE = 15

PPO_MODEL_PATH = "../../trained_models/ppo.pth"  # adjust if running from a different working directory

if not os.path.exists(PPO_MODEL_PATH):
    raise FileNotFoundError(
        f"{PPO_MODEL_PATH} not found. Update PPO_MODEL_PATH to point at the trained ppo.pth file "
        "(see trained_models/ppo.pth or models/rl/ppo-fold4-2026-08-13/model.pth)."
    )


class ActorCritic(nn.Module):
    """PPO Actor-Critic network -- architecture copied from PPO.ipynb so the
    state_dict in ppo.pth loads correctly."""

    def __init__(self, state_size):
        super().__init__()
        self.state_size = state_size
        self.action_size = ACTION_SIZE
        self.shared = nn.Sequential(
            nn.Linear(state_size, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
        )
        self.actor = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, self.action_size), nn.Softmax(dim=-1),
        )
        self.critic = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, 1),
        )
        self.to(DEVICE)

    def forward(self, state):
        if not torch.is_tensor(state):
            state = torch.FloatTensor(state)
        state = state.to(DEVICE)
        features = self.shared(state)
        action_probabilities = self.actor(features)
        state_value = self.critic(features)
        return action_probabilities, state_value


ppo_model = ActorCritic(STATE_SIZE)
checkpoint = torch.load(PPO_MODEL_PATH, map_location=DEVICE)
ppo_model.actor.load_state_dict.__self__  # no-op, keeps flake8 quiet about unused import patterns
ppo_model.load_state_dict(checkpoint["actor_state_dict"])
ppo_model.critic.load_state_dict(checkpoint["critic_state_dict"])
ppo_model.eval()

print("=" * 60)
print("PPO Model Loaded Successfully")
print("=" * 60)
print(f"Model Path : {PPO_MODEL_PATH}")
print(f"Device     : {DEVICE}")
print("=" * 60)

### 7. Prediction Function (used by both LIME and evaluation)

Takes RAW 10-feature rows (as LIME will generate via perturbation), preprocesses each one with `to_trained_representation()`, and returns class probabilities from the real PPO model. This keeps LIME operating against the exact same model and preprocessing as the deployed prediction.

In [ ]:
def predict_probability_from_raw(raw_feature_rows: np.ndarray) -> np.ndarray:
    """
    Parameters
    ----------
    raw_feature_rows : np.ndarray of shape (n_samples, 10)
        Rows in CANONICAL_FEATURE_ORDER, raw (unscaled, non-one-hot) values --
        this is the format LIME's LimeTabularExplainer works with.

    Returns
    -------
    np.ndarray of shape (n_samples, 2) -- class probabilities [P(no disease), P(disease)]
    """
    preprocessed_rows = []
    for row in raw_feature_rows:
        features = dict(zip(CANONICAL_FEATURE_ORDER, row))
        # LIME perturbs continuous AND discretized-categorical columns with
        # small float noise; round categorical columns back to valid ints.
        for key in ("restecg", "slope", "sex", "cp", "exang", "fbs"):
            features[key] = int(round(features[key]))
        preprocessed_rows.append(to_trained_representation(features))

    batch = torch.FloatTensor(np.stack(preprocessed_rows)).to(DEVICE)
    with torch.no_grad():
        action_probabilities, _ = ppo_model(batch)

    return action_probabilities.cpu().numpy()

### 8. LIME Explainer Setup

Trained on a background sample of raw (10-feature) rows so LIME understands realistic feature ranges for perturbation. Uses `research/data/heart_disease_trimmed.csv` (the same cleaned reference dataset the RobustScaler was fitted on).

In [ ]:
REFERENCE_DATA_PATH = "../data/heart_disease_trimmed.csv"

reference_df = pd.read_csv(REFERENCE_DATA_PATH)
background_data = reference_df[CANONICAL_FEATURE_ORDER].to_numpy(dtype=np.float32)

print("Background data shape:", background_data.shape)

categorical_feature_indices = [CANONICAL_FEATURE_ORDER.index(c) for c in ("restecg", "slope", "sex", "cp", "exang", "fbs")]

explainer = LimeTabularExplainer(
    training_data=background_data,
    feature_names=[FEATURE_LABELS[c] for c in CANONICAL_FEATURE_ORDER],
    class_names=["No Heart Disease", "Heart Disease"],
    categorical_features=categorical_feature_indices,
    mode="classification",
    discretize_continuous=True,
)

print("LIME explainer initialized.")

### 9. Generate a LIME Explanation for One Patient

In [ ]:
def explain_patient(features: dict, num_features: int = 10):
    """
    Parameters
    ----------
    features : dict
        The 10 raw canonical features for one patient.

    Returns
    -------
    lime.explanation.Explanation
    """
    raw_row = np.array([features[c] for c in CANONICAL_FEATURE_ORDER], dtype=np.float32)
    return explainer.explain_instance(raw_row, predict_probability_from_raw, num_features=num_features)


# Example patient (values are RAW, exactly as a patient would enter them)
example_patient = {
    "thalach": 150, "restecg": 2, "oldpeak": 2.3, "slope": 0, "age": 63,
    "sex": 1, "cp": 0, "exang": 0, "trestbps": 145, "fbs": 1,
}

example_probabilities = predict_probability_from_raw(np.array([[example_patient[c] for c in CANONICAL_FEATURE_ORDER]]))[0]
example_prediction = int(np.argmax(example_probabilities))

print("=" * 60)
print("EXAMPLE PATIENT PREDICTION")
print("=" * 60)
print(f"Prediction              : {['No Heart Disease', 'Heart Disease'][example_prediction]}")
print(f"P(No Heart Disease)     : {example_probabilities[0]:.4f}")
print(f"P(Heart Disease)        : {example_probabilities[1]:.4f}")
print("=" * 60)

example_explanation = explain_patient(example_patient)

print("\nLIME Feature Contributions (sorted by magnitude):")
for feature_description, weight in sorted(example_explanation.as_list(), key=lambda x: abs(x[1]), reverse=True):
    direction = "increases risk" if weight > 0 else "decreases risk"
    print(f"  {feature_description:45s} {weight:+.4f}  ({direction})")

### 10. LIME Explainability Metrics -- Overview

These metrics evaluate the QUALITY of the LIME explanations themselves, not the RL model's classification performance (accuracy/precision/etc. belong to `PPO.ipynb`'s own evaluation, not here). Metrics computed below:

| Metric | What it measures |
|---|---|
| **Fidelity** | How well LIME's local linear surrogate approximates the real PPO model in the neighborhood of each explained instance (LIME's own R² score). |
| **Accuracy Gain** | How much better the surrogate's own predictions are than a naive majority-class baseline -- i.e. does the simplified explanation carry real signal? |
| **Agreement** | How often the surrogate's predicted class matches the real PPO model's predicted class for the same input. |
| **Stability** | How similar the feature-contribution weights are across repeated LIME runs on the SAME instance (cosine similarity) -- low stability means the explanation is noisy/random. |
| **Sparsity** | How concentrated the contribution weights are on a few features vs. spread thinly across all of them (Gini coefficient) -- higher sparsity means a more human-readable explanation. |
| **Deletion AUC** | Ranks features by LIME weight, then progressively replaces the most important ones with a baseline value and tracks how fast the real model's confidence in its own prediction collapses. Lower is better (a faithful ranking causes a fast collapse). |
| **Insertion AUC** | The mirror of Deletion: starts from an all-baseline input and progressively restores the most important features first, tracking how fast confidence climbs back up. Higher is better. |
| **Reward Improvement** | Using the same reward rule as RL training (`HeartDiseaseEnvironment.step()` in `PPO.ipynb`: +1 if the chosen action matches the true label, else -1), compares the real PPO policy's average reward against the average reward if decisions were made using LIME's local surrogate instead -- quantifies what is lost by approximating the policy with a simple interpretable model. |

### 11. Evaluation Sample

Metrics below are computed over a random sample of patients from the cleaned reference dataset (`research/data/heart_disease_trimmed.csv`), since running LIME (which itself trains hundreds of perturbed-sample predictions per explanation) is far more expensive than a single model forward pass. Increase `NUM_EVAL_PATIENTS` for a more precise estimate at the cost of runtime.

In [ ]:
NUM_EVAL_PATIENTS = 30
NUM_SAMPLES_PER_EXPLANATION = 300  # LIME's internal perturbation count per explain_instance call
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)
eval_indices = rng.choice(len(reference_df), size=min(NUM_EVAL_PATIENTS, len(reference_df)), replace=False)
eval_df = reference_df.iloc[eval_indices].reset_index(drop=True)

print(f"Evaluating LIME on {len(eval_df)} sampled patients.")

### 12. Fidelity, Agreement, and Accuracy Gain

For each sampled patient:
1. Get the real PPO model's predicted class (`predicted_class`).
2. Run LIME for that predicted class -- `explainer.explain_instance(..., labels=(predicted_class,))` returns `exp.score` (the local surrogate's R² -- **Fidelity**) and `exp.local_pred` (the surrogate's own probability estimate for that class).
3. Round the surrogate's probability to a class label and compare it against the real model's predicted class (**Agreement**) and the true label (used for **Accuracy Gain**).

In [ ]:
fidelity_scores = []
agreement_flags = []
surrogate_predictions = []
true_labels = []
real_model_predictions = []

for _, row in eval_df.iterrows():
    features = {c: row[c] for c in CANONICAL_FEATURE_ORDER}
    for key in ("restecg", "slope", "sex", "cp", "exang", "fbs"):
        features[key] = int(features[key])

    raw_row = np.array([features[c] for c in CANONICAL_FEATURE_ORDER], dtype=np.float32)

    real_probabilities = predict_probability_from_raw(np.array([raw_row]))[0]
    predicted_class = int(np.argmax(real_probabilities))

    explanation = explainer.explain_instance(
        raw_row, predict_probability_from_raw,
        labels=(predicted_class,), num_features=10, num_samples=NUM_SAMPLES_PER_EXPLANATION,
    )

    fidelity_scores.append(explanation.score)

    # local_pred is the surrogate's own estimate of P(predicted_class); a
    # value >= 0.5 means the surrogate agrees the predicted_class is more likely.
    surrogate_agrees_with_class = int(explanation.local_pred[0] >= 0.5)
    surrogate_predicted_class = predicted_class if surrogate_agrees_with_class else (1 - predicted_class)

    agreement_flags.append(surrogate_predicted_class == predicted_class)
    surrogate_predictions.append(surrogate_predicted_class)
    real_model_predictions.append(predicted_class)
    true_labels.append(int(row["target"]))

fidelity_scores = np.array(fidelity_scores, dtype=np.float64)
agreement_flags = np.array(agreement_flags, dtype=bool)
surrogate_predictions = np.array(surrogate_predictions, dtype=np.int64)
true_labels = np.array(true_labels, dtype=np.int64)
real_model_predictions = np.array(real_model_predictions, dtype=np.int64)

mean_fidelity = float(np.mean(fidelity_scores))
agreement_rate = float(np.mean(agreement_flags))

# Accuracy Gain: surrogate accuracy vs. true labels, minus a majority-class
# baseline computed on the SAME evaluation sample.
surrogate_accuracy = float(np.mean(surrogate_predictions == true_labels))
majority_class = int(np.round(np.mean(true_labels)))
baseline_accuracy = float(np.mean(np.full_like(true_labels, majority_class) == true_labels))
accuracy_gain = surrogate_accuracy - baseline_accuracy

print("=" * 60)
print("FIDELITY, AGREEMENT, ACCURACY GAIN")
print("=" * 60)
print(f"Mean Fidelity (R^2 of local surrogate)   : {mean_fidelity:.4f}")
print(f"Agreement (surrogate vs. real model)     : {agreement_rate:.4f}")
print(f"Surrogate Accuracy (vs. true labels)     : {surrogate_accuracy:.4f}")
print(f"Majority-Class Baseline Accuracy         : {baseline_accuracy:.4f}")
print(f"Accuracy Gain (surrogate - baseline)     : {accuracy_gain:+.4f}")
print("=" * 60)

### 13. Stability

Re-runs LIME multiple times on the SAME patient (with different internal random perturbations each time) and measures how similar the resulting per-feature weight vectors are, via pairwise cosine similarity. A stable explanation method should return nearly identical feature weights each time for the same input; low cosine similarity indicates the explanation is sensitive to sampling noise and should not be over-interpreted.

In [ ]:
NUM_STABILITY_PATIENTS = 10   # subset of eval_df, since this repeats LIME NUM_STABILITY_RUNS times each
NUM_STABILITY_RUNS = 5

def explanation_weight_vector(explanation, label, num_total_features):
    """Turn LIME's sparse (feature_index, weight) list into a dense vector
    of length num_total_features, so repeated runs (which may select a
    different subset of top features) can be compared directly."""
    vector = np.zeros(num_total_features, dtype=np.float64)
    for feature_index, weight in explanation.local_exp[label]:
        vector[feature_index] = weight
    return vector


def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


stability_scores = []
stability_subset = eval_df.iloc[:min(NUM_STABILITY_PATIENTS, len(eval_df))]

for _, row in stability_subset.iterrows():
    features = {c: row[c] for c in CANONICAL_FEATURE_ORDER}
    for key in ("restecg", "slope", "sex", "cp", "exang", "fbs"):
        features[key] = int(features[key])
    raw_row = np.array([features[c] for c in CANONICAL_FEATURE_ORDER], dtype=np.float32)

    real_probabilities = predict_probability_from_raw(np.array([raw_row]))[0]
    predicted_class = int(np.argmax(real_probabilities))

    weight_vectors = []
    for _run in range(NUM_STABILITY_RUNS):
        explanation = explainer.explain_instance(
            raw_row, predict_probability_from_raw,
            labels=(predicted_class,), num_features=len(CANONICAL_FEATURE_ORDER),
            num_samples=NUM_SAMPLES_PER_EXPLANATION,
        )
        weight_vectors.append(explanation_weight_vector(explanation, predicted_class, len(CANONICAL_FEATURE_ORDER)))

    pairwise_similarities = [
        cosine_similarity(weight_vectors[i], weight_vectors[j])
        for i in range(len(weight_vectors))
        for j in range(i + 1, len(weight_vectors))
    ]
    stability_scores.append(np.mean(pairwise_similarities))

mean_stability = float(np.mean(stability_scores))

print("=" * 60)
print("STABILITY")
print("=" * 60)
print(f"Patients evaluated       : {len(stability_subset)}")
print(f"Repeated runs per patient: {NUM_STABILITY_RUNS}")
print(f"Mean Stability (cosine)  : {mean_stability:.4f}")
print("=" * 60)

### 14. Sparsity

Computes the Gini coefficient of the absolute feature-contribution weights for each explained patient (from Section 12). A Gini coefficient of 0 means every feature contributed equally (a dense, hard-to-read explanation); a value approaching 1 means the explanation is dominated by very few features (a sparse, easily human-readable explanation).

In [ ]:
def gini_coefficient(values):
    """Gini coefficient of a non-negative array. 0 = perfectly uniform, ~1 = maximally concentrated."""
    values = np.sort(np.abs(np.asarray(values, dtype=np.float64)))
    n = len(values)
    if n == 0 or np.sum(values) == 0:
        return 0.0
    cumulative = np.cumsum(values)
    return float((n + 1 - 2 * np.sum(cumulative) / cumulative[-1]) / n)


sparsity_scores = []

for _, row in eval_df.iterrows():
    features = {c: row[c] for c in CANONICAL_FEATURE_ORDER}
    for key in ("restecg", "slope", "sex", "cp", "exang", "fbs"):
        features[key] = int(features[key])
    raw_row = np.array([features[c] for c in CANONICAL_FEATURE_ORDER], dtype=np.float32)

    real_probabilities = predict_probability_from_raw(np.array([raw_row]))[0]
    predicted_class = int(np.argmax(real_probabilities))

    explanation = explainer.explain_instance(
        raw_row, predict_probability_from_raw,
        labels=(predicted_class,), num_features=len(CANONICAL_FEATURE_ORDER),
        num_samples=NUM_SAMPLES_PER_EXPLANATION,
    )
    weights = [weight for _, weight in explanation.local_exp[predicted_class]]
    sparsity_scores.append(gini_coefficient(weights))

mean_sparsity = float(np.mean(sparsity_scores))

print("=" * 60)
print("SPARSITY")
print("=" * 60)
print(f"Mean Sparsity (Gini coefficient): {mean_sparsity:.4f}")
print("=" * 60)

### 15. Deletion and Insertion AUC (Faithfulness)

Fidelity, Agreement, and Sparsity all evaluate the explanation's *surrogate model* or its *shape* -- none of them directly test whether the features LIME calls "important" are actually the features the real PPO model relies on. The deletion/insertion test (Petsiuk et al., 2018, "RISE"; Samek et al., 2016, AOPC; adapted to tabular data by replacing pixels with feature values) answers that directly, by perturbing the REAL model's input -- not the surrogate -- and watching how its own output probability responds:

- **Deletion**: rank the 10 features by |LIME weight|, most important first. Starting from the patient's real row, replace features one at a time (most important first) with a neutral baseline (the reference dataset's per-feature median/mode), re-querying the real PPO model after each replacement. Track P(predicted_class) after each step. A faithful ranking should cause this probability to collapse quickly -- so **lower area under this curve is better**.
- **Insertion**: the mirror test. Start from an all-baseline row (every feature at its reference median/mode) and restore features one at a time, most important first, tracking how quickly P(predicted_class) climbs back toward the real prediction. **Higher area under this curve is better**.
- Both curves are normalized to the fraction of features removed/restored (0 to 1 on the x-axis) so AUC is comparable across patients and directly comparable to each other; both are computed on the SAME evaluation sample as Sections 12-14, against the REAL model (not the LIME surrogate), which is what makes this a faithfulness check rather than a surrogate-quality check.
- The cell below prints and saves EACH evaluated patient's own confidence, predicted class, Deletion AUC, and Insertion AUC (`lime_eval_deletion_insertion_per_patient.csv`) BEFORE averaging them into the single `mean_deletion_auc` / `mean_insertion_auc` reported in the Section 17 summary -- so you can see exactly which patients had a faithful vs. unfaithful explanation, not just the population-level average.

In [ ]:
# Per-feature baseline value used to "remove" a feature: median for the four
# continuous features, mode for the six categorical/binary features -- both
# computed on the same cleaned reference dataset already loaded as reference_df.
FEATURE_BASELINE_VALUES = {}
for _col in CANONICAL_FEATURE_ORDER:
    if _col in ("restecg", "slope", "sex", "cp", "exang", "fbs"):
        FEATURE_BASELINE_VALUES[_col] = float(reference_df[_col].mode().iloc[0])
    else:
        FEATURE_BASELINE_VALUES[_col] = float(reference_df[_col].median())

print("Feature baseline values (median/mode):")
for _col, _val in FEATURE_BASELINE_VALUES.items():
    print(f"  {_col:10s} {_val}")


def auc_of_curve(y_values):
    """Trapezoidal area under a curve sampled at len(y_values) evenly-spaced
    x-points across [0, 1] (0 = no features removed/restored, 1 = all removed/restored)."""
    if len(y_values) < 2:
        return float(y_values[0]) if y_values else 0.0
    x_values = np.linspace(0.0, 1.0, num=len(y_values))
    return float(np.trapz(y_values, x_values))


def deletion_curve(raw_row, feature_rank, predicted_class):
    """P(predicted_class) from the REAL model as features are progressively
    replaced with their baseline value, most-important-first per feature_rank
    (a list of feature indices into CANONICAL_FEATURE_ORDER, descending importance)."""
    working_row = raw_row.copy()
    probabilities = [predict_probability_from_raw(np.array([working_row]))[0][predicted_class]]
    for feature_index in feature_rank:
        feature_name = CANONICAL_FEATURE_ORDER[feature_index]
        working_row[feature_index] = FEATURE_BASELINE_VALUES[feature_name]
        probabilities.append(predict_probability_from_raw(np.array([working_row]))[0][predicted_class])
    return probabilities


def insertion_curve(raw_row, feature_rank, predicted_class):
    """Mirror of deletion_curve: starts fully baselined, restores the real
    value of the most-important feature first."""
    working_row = np.array(
        [FEATURE_BASELINE_VALUES[c] for c in CANONICAL_FEATURE_ORDER], dtype=np.float32
    )
    probabilities = [predict_probability_from_raw(np.array([working_row]))[0][predicted_class]]
    for feature_index in feature_rank:
        working_row[feature_index] = raw_row[feature_index]
        probabilities.append(predict_probability_from_raw(np.array([working_row]))[0][predicted_class])
    return probabilities


deletion_aucs = []
insertion_aucs = []
per_patient_records = []  # one row per evaluated patient, before averaging

for patient_index, (_, row) in enumerate(eval_df.iterrows()):
    features = {c: row[c] for c in CANONICAL_FEATURE_ORDER}
    for key in ("restecg", "slope", "sex", "cp", "exang", "fbs"):
        features[key] = int(features[key])
    raw_row = np.array([features[c] for c in CANONICAL_FEATURE_ORDER], dtype=np.float32)

    real_probabilities = predict_probability_from_raw(np.array([raw_row]))[0]
    predicted_class = int(np.argmax(real_probabilities))
    original_confidence = float(real_probabilities[predicted_class])

    explanation = explainer.explain_instance(
        raw_row, predict_probability_from_raw,
        labels=(predicted_class,), num_features=len(CANONICAL_FEATURE_ORDER),
        num_samples=NUM_SAMPLES_PER_EXPLANATION,
    )
    weight_vector = explanation_weight_vector(explanation, predicted_class, len(CANONICAL_FEATURE_ORDER))
    feature_rank = list(np.argsort(-np.abs(weight_vector)))  # descending |weight|

    patient_deletion_curve = deletion_curve(raw_row, feature_rank, predicted_class)
    patient_insertion_curve = insertion_curve(raw_row, feature_rank, predicted_class)
    patient_deletion_auc = auc_of_curve(patient_deletion_curve)
    patient_insertion_auc = auc_of_curve(patient_insertion_curve)

    deletion_aucs.append(patient_deletion_auc)
    insertion_aucs.append(patient_insertion_auc)

    per_patient_records.append({
        "Patient": patient_index,
        "Predicted Class": ["No Heart Disease", "Heart Disease"][predicted_class],
        "Original Confidence": original_confidence,
        "Deletion AUC": patient_deletion_auc,
        "Insertion AUC": patient_insertion_auc,
        # Full step-by-step confidence curves, in case per-step inspection
        # is needed (e.g. plotting one patient's collapse/recovery curve).
        "Deletion Curve": patient_deletion_curve,
        "Insertion Curve": patient_insertion_curve,
    })

mean_deletion_auc = float(np.mean(deletion_aucs))
mean_insertion_auc = float(np.mean(insertion_aucs))

# Per-patient breakdown -- shows each patient's own confidence-based AUC
# before it gets collapsed into the single averaged number below.
per_patient_df = pd.DataFrame(per_patient_records)

print("=" * 78)
print("PER-PATIENT DELETION / INSERTION AUC (before averaging)")
print("=" * 78)
print(
    per_patient_df[["Patient", "Predicted Class", "Original Confidence", "Deletion AUC", "Insertion AUC"]]
    .to_string(index=False, float_format=lambda v: f"{v:.4f}")
)
print("=" * 78)

print("\n" + "=" * 60)
print("DELETION / INSERTION (FAITHFULNESS) -- AVERAGE ACROSS ALL PATIENTS")
print("=" * 60)
print(f"Patients evaluated                          : {len(per_patient_df)}")
print(f"Mean Deletion AUC (lower = more faithful)   : {mean_deletion_auc:.4f}")
print(f"Mean Insertion AUC (higher = more faithful) : {mean_insertion_auc:.4f}")
print("=" * 60)

# Saved separately from the final summary CSV (Section 17) so the raw
# per-patient breakdown -- including full confidence curves -- remains
# available for inspection/plotting, not just the averaged scalars.
per_patient_df.drop(columns=["Deletion Curve", "Insertion Curve"]).to_csv(
    "lime_eval_deletion_insertion_per_patient.csv", index=False
)
print("\nSaved: lime_eval_deletion_insertion_per_patient.csv")

### 16. Plotting One Patient's Deletion / Insertion Curve

The per-patient table above only shows each patient's AUC (the area under their curve) -- not the curve's actual shape. Plotting one patient's curve directly shows WHY that patient got the AUC it did: a curve that drops off a cliff after 1-2 features gives a very different visual impression than one that fades out gradually, even if their AUCs end up similar.

`PLOT_PATIENT_INDEX` selects which row of `eval_df` / `per_patient_df` to plot (matches the "Patient" column in the printed table above -- change it to inspect a different patient, e.g. one with an unusually high or low Deletion AUC).

In [ ]:
PLOT_PATIENT_INDEX = 0  # change to any index printed in the per-patient table above

plotted_record = per_patient_records[PLOT_PATIENT_INDEX]
deletion_curve_to_plot = plotted_record["Deletion Curve"]
insertion_curve_to_plot = plotted_record["Insertion Curve"]
num_steps = len(deletion_curve_to_plot)
x_axis = np.linspace(0.0, 1.0, num=num_steps)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(x_axis, deletion_curve_to_plot, marker="o", color="tab:red")
axes[0].fill_between(x_axis, deletion_curve_to_plot, alpha=0.2, color="tab:red")
axes[0].set_title(f"Deletion Curve -- Patient {plotted_record['Patient']}\nAUC = {plotted_record['Deletion AUC']:.4f} (lower is better)")
axes[0].set_xlabel("Fraction of features removed")
axes[0].set_ylabel(f"P({plotted_record['Predicted Class']})")
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

axes[1].plot(x_axis, insertion_curve_to_plot, marker="o", color="tab:green")
axes[1].fill_between(x_axis, insertion_curve_to_plot, alpha=0.2, color="tab:green")
axes[1].set_title(f"Insertion Curve -- Patient {plotted_record['Patient']}\nAUC = {plotted_record['Insertion AUC']:.4f} (higher is better)")
axes[1].set_xlabel("Fraction of features restored")
axes[1].set_ylabel(f"P({plotted_record['Predicted Class']})")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

fig.suptitle(f"Patient {plotted_record['Patient']} -- Predicted: {plotted_record['Predicted Class']} "
             f"(original confidence {plotted_record['Original Confidence']:.4f})")
plt.tight_layout()
plt.show()

print("=" * 60)
print(f"PATIENT {plotted_record['Patient']} -- DELETION/INSERTION CURVE VALUES")
print("=" * 60)
print(f"Predicted class       : {plotted_record['Predicted Class']}")
print(f"Original confidence   : {plotted_record['Original Confidence']:.4f}")
print(f"Deletion AUC          : {plotted_record['Deletion AUC']:.4f}")
print(f"Insertion AUC         : {plotted_record['Insertion AUC']:.4f}")
print("-" * 60)
print(f"{'Step':>5s} {'Deletion P':>12s} {'Insertion P':>12s}")
for step_index in range(num_steps):
    print(f"{step_index:>5d} {deletion_curve_to_plot[step_index]:>12.4f} {insertion_curve_to_plot[step_index]:>12.4f}")
print("=" * 60)

### 17. Reward Improvement

Uses the SAME reward rule the PPO agent was trained with (`HeartDiseaseEnvironment.step()` in `PPO.ipynb`: reward = +1 if the chosen action equals the true label, else -1). Compares the average reward of the real trained PPO policy against the average reward that would result from using LIME's local linear surrogate's own decision instead. A positive value means the trained policy outperforms the simplified explanation (expected, since PPO is a full nonlinear policy and LIME's surrogate is a local linear approximation) -- this quantifies the reward cost of relying on the interpretable approximation rather than the real policy.

In [ ]:
def step_reward(action, true_label):
    """Matches HeartDiseaseEnvironment.step() in PPO.ipynb exactly."""
    return 1 if action == true_label else -1


ppo_rewards = [step_reward(pred, true) for pred, true in zip(real_model_predictions, true_labels)]
surrogate_rewards = [step_reward(pred, true) for pred, true in zip(surrogate_predictions, true_labels)]

mean_ppo_reward = float(np.mean(ppo_rewards))
mean_surrogate_reward = float(np.mean(surrogate_rewards))
reward_improvement = mean_ppo_reward - mean_surrogate_reward

print("=" * 60)
print("REWARD IMPROVEMENT")
print("=" * 60)
print(f"Mean Reward -- Real PPO Policy       : {mean_ppo_reward:+.4f}")
print(f"Mean Reward -- LIME Local Surrogate  : {mean_surrogate_reward:+.4f}")
print(f"Reward Improvement (PPO - Surrogate) : {reward_improvement:+.4f}")
print("=" * 60)

### 18. Summary

In [ ]:
lime_metrics_summary = pd.DataFrame([{
    "Fidelity": mean_fidelity,
    "Accuracy Gain": accuracy_gain,
    "Agreement": agreement_rate,
    "Stability": mean_stability,
    "Sparsity": mean_sparsity,
    "Deletion AUC": mean_deletion_auc,
    "Insertion AUC": mean_insertion_auc,
    "Reward Improvement": reward_improvement,
}])

print("=" * 70)
print("LIME EXPLAINABILITY METRICS SUMMARY")
print("=" * 70)
print(lime_metrics_summary.to_string(index=False))
print("=" * 70)

lime_metrics_summary.to_csv("lime_eval_explainability_metrics.csv", index=False)
print("\nSaved: lime_eval_explainability_metrics.csv")